In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP03 - TP Services
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Did the unit use third party intermediaries or subcontractors during the assessment 
   period to perform marketing, business development, sales, consulting, or brokerage?
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. RISK FLAG LOGIC: Flags engagements as high risk if they meet either condition:
      - KPI_Number = '8.5' AND Value = '100'
      - primary_product_serv matches the predefined list of 14 categories. (Converted 
        to UPPER() for case-insensitive matching).
   3. COMMA EXCEPTION HANDLING: Protects specific unit names containing commas (e.g., 
      'TECE - Strategy, Change...') using the '[COMMA]' placeholder swap to prevent 
      improper splitting during the explosion phase.
   4. FLATTEN LOBs: Uses LATERAL VIEW explode(split(...)) to expand comma-separated 
      LOB lists. Restores '[COMMA]' to a real ',' after the split.
   5. SAFE MAPPING: Maps expanded LOBs to TPRM_Units using standard LIKE syntax.
   6. DUAL-BRIDGE JOIN: Matches on either the String Name OR the Numeric BusinessID 
      provided in the mapping table's Assessable_Unit_ID column.
   7. FINAL OUTPUT: Binary 'Yes' / 'No' output based on whether any matched high-risk 
      engagements exist for the AU.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 2. Grab valid TPRM Mapping Strings (Accommodates mixed ID/Name format)
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 3. Extract columns and apply Comma Protection Dictionary
    SELECT 
        EngagementNumber,
        KPI_Number,
        Value,
        primary_product_serv,
        
        -- EXCEPTION DICTIONARY: Chain replaces to shield internal commas from split()
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
            
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flagged_Engagements AS (
    -- 4. Filter based on KPI 8.5 OR target marketing/printing categories
    SELECT DISTINCT
        EngagementNumber,
        safe_owning_lob,
        safe_lob_using
    FROM Third_Party_Risk_Data
    WHERE 
        (TRY_CAST(TRIM(CAST(KPI_Number AS STRING)) AS DOUBLE) = 8.5 AND TRY_CAST(TRIM(CAST(Value AS STRING)) AS DOUBLE) = 100.0)
        OR 
        UPPER(TRIM(primary_product_serv)) IN (
            'MARKETING MEDIA SERVICES', 'MARKETING AND DISTRIBUTION', 'DIRECT MARKETING PRINT SERVICE',
            'PRINT ADVERTISING', 'PRINTING', 'MANAGEMENT AND BUSINESS PROFESSIONALS AND ADMINISTRATIVE SERVICES',
            'MARKETING ANALYSIS', 'PUBLICITY AND MARKETING SUPPORT SERVICES', 'MARKETING COMMUNICATIONS AGENCIES',
            'ADVERTISING AGENCY SERVICES', 'BUSINESS FORMS OR QUESTIONNAIRES', 'SPECIALIZED WAREHOUSING AND STORAGE',
            'STATIONERY OR BUSINESS FORM PRINTING', 'PUBLISHED PRODUCTS'
        )
),

Flattened_LOBs AS (
    -- 5. Explode comma-separated strings into individual rows and restore commas
    SELECT 
        EngagementNumber, 
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 6. Map expanded rows to AU using Dual-Bridge and count matches
    SELECT 
        mast.BusinessID,
        -- DEDUPLICATION: Count distinct engagements to avoid double-counting expanded rows
        COUNT(DISTINCT f.EngagementNumber) AS Match_Count
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    
    -- DUAL-BRIDGE JOIN: Supports mapping table's mix of Numeric ID or String Name
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
        
    GROUP BY mast.BusinessID
)

-- 7. Final Output: Strictly structured per Master Anchor
SELECT 
    a.BusinessID, 
    a.AU_Name, 
    a.Business_Segment,
    'TP03' AS QuestionID,               
    
    CASE 
        WHEN COALESCE(e.Match_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP03 - AU Level Calculation Review
   PURPOSE: One row per AU showing the engagements, bridge keys, and the rule path
   behind the final Yes / No response.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        KPI_Number,
        Value,
        primary_product_serv,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flagged_Engagements AS (
    SELECT DISTINCT
        EngagementNumber,
        safe_owning_lob,
        safe_lob_using,
        CASE WHEN TRY_CAST(TRIM(CAST(KPI_Number AS STRING)) AS DOUBLE) = 8.5 AND TRY_CAST(TRIM(CAST(Value AS STRING)) AS DOUBLE) = 100.0 THEN 1 ELSE 0 END AS KPI_85_100_Flag,
        CASE WHEN UPPER(TRIM(primary_product_serv)) IN (
            'MARKETING MEDIA SERVICES', 'MARKETING AND DISTRIBUTION', 'DIRECT MARKETING PRINT SERVICE',
            'PRINT ADVERTISING', 'PRINTING', 'MANAGEMENT AND BUSINESS PROFESSIONALS AND ADMINISTRATIVE SERVICES',
            'MARKETING ANALYSIS', 'PUBLICITY AND MARKETING SUPPORT SERVICES', 'MARKETING COMMUNICATIONS AGENCIES',
            'ADVERTISING AGENCY SERVICES', 'BUSINESS FORMS OR QUESTIONNAIRES', 'SPECIALIZED WAREHOUSING AND STORAGE',
            'STATIONERY OR BUSINESS FORM PRINTING', 'PUBLISHED PRODUCTS'
        ) THEN 1 ELSE 0 END AS Product_Service_Flag
    FROM Third_Party_Risk_Data
    WHERE 
        (TRY_CAST(TRIM(CAST(KPI_Number AS STRING)) AS DOUBLE) = 8.5 AND TRY_CAST(TRIM(CAST(Value AS STRING)) AS DOUBLE) = 100.0)
        OR 
        UPPER(TRIM(primary_product_serv)) IN (
            'MARKETING MEDIA SERVICES', 'MARKETING AND DISTRIBUTION', 'DIRECT MARKETING PRINT SERVICE',
            'PRINT ADVERTISING', 'PRINTING', 'MANAGEMENT AND BUSINESS PROFESSIONALS AND ADMINISTRATIVE SERVICES',
            'MARKETING ANALYSIS', 'PUBLICITY AND MARKETING SUPPORT SERVICES', 'MARKETING COMMUNICATIONS AGENCIES',
            'ADVERTISING AGENCY SERVICES', 'BUSINESS FORMS OR QUESTIONNAIRES', 'SPECIALIZED WAREHOUSING AND STORAGE',
            'STATIONERY OR BUSINESS FORM PRINTING', 'PUBLISHED PRODUCTS'
        )
),

Flattened_LOBs AS (
    SELECT 
        EngagementNumber,
        KPI_85_100_Flag,
        Product_Service_Flag,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Resolved_Bridge AS (
    SELECT 
        mast.BusinessID,
        mast.AU_Name,
        mast.Business_Segment,
        mast.In_ABAC_AU_List,
        f.EngagementNumber,
        f.KPI_85_100_Flag,
        f.Product_Service_Flag,
        f.Expanded_LOB,
        map.TPRM_Units,
        map.Mapping_ID_Or_Name
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
),

AU_Debug AS (
    SELECT 
        BusinessID,
        MAX(AU_Name) AS AU_Name,
        MAX(Business_Segment) AS Business_Segment,
        MAX(In_ABAC_AU_List) AS In_ABAC_AU_List,
        COUNT(DISTINCT EngagementNumber) AS Distinct_Engagement_Count,
        COUNT(DISTINCT CASE WHEN KPI_85_100_Flag = 1 THEN EngagementNumber END) AS KPI_85_100_Engagement_Count,
        COUNT(DISTINCT CASE WHEN Product_Service_Flag = 1 THEN EngagementNumber END) AS Product_Service_Engagement_Count,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(TPRM_Units))) AS Matched_TPRM_Units_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Mapping_ID_Or_Name))) AS Mapping_Bridge_Value_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(EngagementNumber))) AS Engagement_List
    FROM Resolved_Bridge
    GROUP BY BusinessID
)

SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'TP03' AS QuestionID,
    CASE WHEN COALESCE(d.Distinct_Engagement_Count, 0) > 0 THEN 'Yes' ELSE 'No' END AS Response,
    COALESCE(d.Matched_TPRM_Units_List, 'n/a') AS Matched_TPRM_Units_List,
    COALESCE(d.Mapping_Bridge_Value_List, 'n/a') AS Mapping_Bridge_Value_List,
    COALESCE(d.Engagement_List, 'n/a') AS Engagement_List,
    COALESCE(d.Distinct_Engagement_Count, 0) AS Distinct_Engagement_Count,
    COALESCE(d.KPI_85_100_Engagement_Count, 0) AS KPI_85_100_Engagement_Count,
    COALESCE(d.Product_Service_Engagement_Count, 0) AS Product_Service_Engagement_Count,
    CASE 
        WHEN COALESCE(d.Distinct_Engagement_Count, 0) > 0 THEN CONCAT(
            'Yes because ', CAST(d.Distinct_Engagement_Count AS STRING), ' distinct flagged engagement(s) mapped to this AU. KPI 8.5/100 engagements=',
            CAST(d.KPI_85_100_Engagement_Count AS STRING), ', product/service engagements=', CAST(d.Product_Service_Engagement_Count AS STRING)
        )
        ELSE NULL
    END AS Why_Yes,
    CASE 
        WHEN COALESCE(d.Distinct_Engagement_Count, 0) = 0 THEN 'No because no flagged engagement bridged to this AU'
        ELSE NULL
    END AS Why_No,
    'Any bridged engagement that meets KPI 8.5 = 100 or the approved product/service list makes the AU response Yes.' AS Debug_Reason,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN AU_Debug d
    ON mast.BusinessID = d.BusinessID
ORDER BY mast.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE 2: TP03 - KPI 8.5 / 100 Bridge Audit
   PURPOSE: Shows where KPI 8.5 = 100 rows are being lost: missing LOBs, no TPRM
   unit match, no in-scope master AU bridge, or fully resolved.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

KPI_85_100_Rows AS (
    SELECT DISTINCT
        EngagementNumber,
        KPI_Number,
        Value,
        primary_product_serv,
        owning_lob AS Raw_Owning_LOB,
        lob_using AS Raw_Lob_Using,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND TRY_CAST(TRIM(CAST(KPI_Number AS STRING)) AS DOUBLE) = 8.5
      AND TRY_CAST(TRIM(CAST(Value AS STRING)) AS DOUBLE) = 100.0
),

Expanded_KPI_Rows AS (
    SELECT
        k.EngagementNumber,
        k.KPI_Number,
        k.Value,
        k.primary_product_serv,
        k.Raw_Owning_LOB,
        k.Raw_Lob_Using,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM KPI_85_100_Rows k
    LATERAL VIEW OUTER explode(split(concat_ws(',', k.safe_owning_lob, k.safe_lob_using), ',')) AS exploded_lob
),

Resolved_Audit AS (
    SELECT
        e.EngagementNumber,
        e.KPI_Number,
        e.Value,
        e.primary_product_serv,
        e.Raw_Owning_LOB,
        e.Raw_Lob_Using,
        e.Expanded_LOB,
        map.TPRM_Units,
        map.Mapping_ID_Or_Name,
        mast.BusinessID AS Bridged_BusinessID,
        mast.AU_Name AS Bridged_AU_Name,
        CASE
            WHEN COALESCE(TRIM(e.Raw_Owning_LOB), '') = ''
             AND COALESCE(TRIM(e.Raw_Lob_Using), '') = '' THEN 'KPI row has no LOB values'
            WHEN COALESCE(TRIM(e.Expanded_LOB), '') = '' THEN 'KPI row produced no explodable LOB'
            WHEN map.TPRM_Units IS NULL THEN 'Expanded LOB did not match any TPRM unit'
            WHEN mast.BusinessID IS NULL THEN 'TPRM mapping did not bridge to in-scope master AU'
            ELSE 'Resolved to in-scope master AU'
        END AS Debug_Stage
    FROM Expanded_KPI_Rows e
    LEFT JOIN Mapped_AUs map
        ON COALESCE(UPPER(e.Expanded_LOB), '') LIKE '%' || UPPER(map.TPRM_Units) || '%'
    LEFT JOIN Master_AUs mast
        ON map.Mapping_ID_Or_Name IS NOT NULL
       AND (
            UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
         OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
       )
)

SELECT DISTINCT
    EngagementNumber,
    KPI_Number,
    Value,
    primary_product_serv,
    Raw_Owning_LOB,
    Raw_Lob_Using,
    Expanded_LOB,
    TPRM_Units,
    Mapping_ID_Or_Name,
    Bridged_BusinessID,
    Bridged_AU_Name,
    Debug_Stage
FROM Resolved_Audit
ORDER BY EngagementNumber, Expanded_LOB, TPRM_Units, Bridged_BusinessID;
